# 08 - Radiomics Pipeline (Path 1)

**Inputs needed:** Raw DICOMs for several patients, `data/labels.csv`, `configs/default.yaml`.
**Outputs produced:** Phase 2 + 3 artefacts (cropped volumes, radiomics CSVs, VaRFS selection JSON, baseline checkpoint and metrics).
**Runtime:** ~5–15 minutes for a small cohort once preprocessing is cached.


Complete radiomics-only path from the design doc:

```
A1/A2 -> B (phase filter + DICOM->NIfTI)
      -> C (HU window + Z-score within liver mask)
      -> D (liver segmentation)
      -> E (liver bounding-box crop)
      -> F (PyRadiomics on full liver volume)
      -> H (VaRFS bootstrap stability filter)
      -> I (stable feature vector + classical ML baseline)
```

End artifacts:
- `data/raw_radiomics_features.csv`
- `data/varfs_filtered_features.csv`
- `data/varfs_selected_features.json`
- `models/saved/radiomics_baseline.pkl`
- `results/baseline_metrics.json` and `results/baseline_scores.csv`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import ensure_dirs, load_config, load_features_config, set_seed
from src.utils.logger import setup_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
features_cfg = load_features_config(ROOT / "configs" / "features.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "radiomics_pipeline.log")
cfg["paths"]

## Stage 1 - Phase 2 preprocessing per patient (B->E)

Skip patients that already have `before_cropped.nii.gz`.

In [ ]:
from tqdm.notebook import tqdm
from src.data.dicom_loader import DICOMLoader
from src.data.liver_segmentation import LiverSegmentor
from src.data.preprocessing import preprocess_volume
from src.data.cropping import crop_patient

raw_dir = Path(cfg["paths"]["raw_dir"])
processed_dir = Path(cfg["paths"]["processed_dir"])
loader = DICOMLoader(raw_dir, cfg["preprocessing"]["target_phase"], processed_dir)
segmentor = LiverSegmentor(
    gpu=bool(cfg["preprocessing"]["liver_segmentation"].get("gpu", True))
)

patient_ids = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())
print(f"Patients to process: {len(patient_ids)}")

errors = {}
for pid in tqdm(patient_ids, desc="Phase 2 preprocessing"):
    out_dir = processed_dir / pid
    if (out_dir / "before_cropped.nii.gz").exists() and (out_dir / "crop_metadata.json").exists():
        continue
    try:
        volume_path = loader.convert_to_nifti(pid)
        mask_path = segmentor.segment(volume_path)
        _, stats = preprocess_volume(volume_path, mask_path, cfg)
        _, _, _ = crop_patient(volume_path.parent, zscore_stats=stats)
    except Exception as exc:
        errors[pid] = str(exc)
errors

## Stage 2 - PyRadiomics extraction (F)

Run on the full liver volume. Output is a (patients x features) matrix.

In [ ]:
from src.features.radiomics_extractor import RadiomicsExtractor

raw_csv = ROOT / "data" / "raw_radiomics_features.csv"
extractor = RadiomicsExtractor(features_cfg["radiomics"])
raw_features = extractor.extract_all(
    processed_dir=processed_dir,
    labels_csv=cfg["paths"]["labels_csv"],
    output_csv=raw_csv,
)
raw_features.shape

In [ ]:
import pandas as pd
feature_classes = pd.Series([c.split("_")[1] if "_" in c else c for c in raw_features.columns if c not in ("patient_id", "label")]).value_counts()
feature_classes.head(15)

## Stage 3 - VaRFS stability filter (H)

Bootstrap resampling -> per-feature stability -> drop highly correlated features.

In [ ]:
from src.features.varfs_selection import VaRFSSelector

varfs_cfg = features_cfg["varfs"]
selector = VaRFSSelector(
    n_bootstrap=int(varfs_cfg["n_bootstrap"]),
    stability_threshold=float(varfs_cfg["stability_threshold"]),
    correlation_threshold=float(varfs_cfg["correlation_threshold"]),
    top_k_per_iter=int(varfs_cfg["top_k_per_iter"]),
    use_robust_scaler=bool(varfs_cfg["use_robust_scaler"]),
    random_state=int(cfg["seed"]),
)
selector.fit(raw_features, raw_features["label"])
filtered = selector.transform(raw_features)
filtered.to_csv(ROOT / "data" / "varfs_filtered_features.csv", index=False)
selector.save_selection(ROOT / "data" / "varfs_selected_features.json")
print(f"Selected: {len(selector.selected_features_)} stable features")

In [ ]:
import matplotlib.pyplot as plt
stability = pd.Series(selector.stability_scores_).sort_values(ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].hist(stability.values, bins=30)
ax[0].axvline(varfs_cfg["stability_threshold"], color="red", linestyle="--", label="threshold")
ax[0].set_title("Stability score distribution")
ax[0].legend()
stability.head(20).plot.barh(ax=ax[1], title="Top 20 stable features")
plt.tight_layout(); plt.show()

## Stage 4 - Classical ML baseline (I)

RandomForest with class_weight=balanced, evaluated through stratified CV.

In [ ]:
from src.features.baseline_classifier import RadiomicsBaseline, save_metrics

baseline = RadiomicsBaseline(features_cfg["baseline"])
metrics = baseline.train(filtered)
model_path = Path(cfg["paths"]["model_save_dir"]) / "radiomics_baseline.pkl"
baseline.save(model_path)
save_metrics(metrics, Path(cfg["paths"]["results_dir"]) / "baseline_metrics.json")
metrics

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc

probs = baseline.predict_proba(filtered)
y = filtered["label"].astype(int).to_numpy()
fpr, tpr, _ = roc_curve(y, probs)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"radiomics (AUC={auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("Radiomics path - ROC")
plt.legend(); plt.show()

scores_df = pd.DataFrame({
    "patient_id": filtered["patient_id"],
    "label": filtered["label"],
    "score": probs,
})
scores_df.to_csv(Path(cfg["paths"]["results_dir"]) / "baseline_scores.csv", index=False)
scores_df.head()

## Stage 5 - Feature importance

In [ ]:
importances = pd.Series(
    baseline.model.feature_importances_,
    index=baseline.feature_columns_,
).sort_values(ascending=False)
importances.head(20).plot.barh(figsize=(7, 6), title="Top 20 RF feature importances")